<a href="https://colab.research.google.com/github/AzamAlzahrani/SDAIA_GenAI/blob/main/assignments/rag01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minimal RAG with Gemini
Add your key in Colab Secrets (key icon, left sidebar) as `GEMINI_API_KEY` and turn on Notebook access. Then run the cells in order.

In [1]:
import numpy as np
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API'))

In [3]:
# 1. Read cities.txt from the Colab files panel
with open('/content/cities.txt', encoding='utf-8') as f:
    text = f.read().strip()

# 2. Chunk: fixed size with overlap
CHUNK_SIZE = 450   # characters per chunk
OVERLAP = 50       # characters repeated from the previous chunk

chunks = []
start = 0
while start < len(text):
    piece = text[start:start + CHUNK_SIZE].strip()
    if piece:
        chunks.append(piece)
    start += CHUNK_SIZE - OVERLAP

print(f'{len(text)} characters -> {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={OVERLAP})')

3998 characters -> 10 chunks (size=450, overlap=50)


In [4]:
# 3. Embed
def embed(texts):
    res = client.models.embed_content(model='gemini-embedding-001', contents=texts)
    v = np.array([e.values for e in res.embeddings])
    return v / np.linalg.norm(v, axis=1, keepdims=True)

vectors = embed(chunks)

#print(vectors.shape)
print(len(vectors[3]))

3072


In [5]:
question = "What is the capital of Japan?"
scores = vectors @ embed([question])[0]
top = np.argsort(scores)[::-1][:3]
print(scores[top])


[0.69060744 0.66695506 0.56190612]


In [6]:
print(vectors)

[[-0.01921333 -0.00425207 -0.00696984 ...  0.00784199 -0.02182287
   0.00461452]
 [-0.01512494 -0.03519994 -0.01401447 ...  0.02091051  0.00472617
  -0.00132   ]
 [ 0.00035952 -0.02858847 -0.01747023 ...  0.00774855  0.00988431
   0.01091668]
 ...
 [-0.01661854  0.00202459 -0.01132254 ... -0.01048638 -0.01597857
   0.00367872]
 [-0.01634478 -0.01055578 -0.01069707 ...  0.0071745  -0.01430644
  -0.00537142]
 [ 0.00650408 -0.00634208 -0.02544479 ...  0.00845552  0.00466487
   0.00510253]]


In [14]:
# 4. Ask: find the most similar chunks, then let Gemini answer
def ask(question, k=3):
    for i in top:
        print(f'{scores[i]:.3f}  {chunks[i][:60]}...')
    context = '\n\n'.join(chunks[i] for i in top)
    reply = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=f'Answer using only this context:\n{context}\n\nQuestion: {question}',
    )
    print('\n' + reply.text)
question = "what is the capital of japan?"

print(embed([question]).shape)
ask(question)

(1, 3072)
0.691  Tokyo is the capital of Japan and lies on the east coast of ...
0.667  Paris is the capital of France and sits on the Seine River i...
0.562  Rome is the capital of Italy and sits on the Tiber River in ...

Based on the provided context, the capital of Japan is Tokyo.
